# Overview

In this notebook, we will go through the task generation pipeline in LIBERO. We will cover the following contents:

1. Retrieve a list of available objects, predicates
 in the codebase
2. Define your own initial state distribution
3. Define your own task goal
4. Generate the pddl file for the task

Now, let's get started!

## 1. Retrieve a list of objects available

In order for the sucess in task generation, we need to make sure that the objects and the predicates (object relations) specified are available in the codebase.

### 本节讲解:为什么要先"检索"可用物体 / 谓词

任务生成的第一步不是写代码,而是**盘点弹药库**:LIBERO 的任务由「物体(objects)」和「谓词(predicates,即物体间的关系)」组合而成。如果场景里写的物体名、或目标里写的谓词名在代码库里不存在,那么**场景加载会失败、或目标永远无法被判定为达成**。

所以第一节的目标很明确:先列出所有**已注册的物体**和**已注册的谓词**,确认你要用的名字真实存在,再往下走。下面两个 cell 分别检索物体和谓词。


In [2]:
from libero.libero.envs.objects import get_object_dict, get_object_fn

# Get a dictionary of all the objects
object_dict = get_object_dict()
print(object_dict)

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /Users/yifengz/workspace/robosuite-master/robosuite/scripts/setup_macros.py (__init__.py:9)


{'alphabet_soup': <class 'libero.libero.envs.objects.hope_objects.AlphabetSoup'>, 'bbq_sauce': <class 'libero.libero.envs.objects.hope_objects.BbqSauce'>, 'butter': <class 'libero.libero.envs.objects.hope_objects.Butter'>, 'cherries': <class 'libero.libero.envs.objects.hope_objects.Cherries'>, 'chocolate_pudding': <class 'libero.libero.envs.objects.hope_objects.ChocolatePudding'>, 'cookies': <class 'libero.libero.envs.objects.hope_objects.Cookies'>, 'corn': <class 'libero.libero.envs.objects.hope_objects.Corn'>, 'cream_cheese': <class 'libero.libero.envs.objects.hope_objects.CreamCheese'>, 'ketchup': <class 'libero.libero.envs.objects.hope_objects.Ketchup'>, 'macaroni_and_cheese': <class 'libero.libero.envs.objects.hope_objects.MacaroniAndCheese'>, 'mayo': <class 'libero.libero.envs.objects.hope_objects.Mayo'>, 'milk': <class 'libero.libero.envs.objects.hope_objects.Milk'>, 'orange_juice': <class 'libero.libero.envs.objects.hope_objects.OrangeJuice'>, 'popcorn': <class 'libero.libero.e

Now you can see all the available objects, you can retrieve the object class by specifying their categories (which are the keys in the dictionary)

### 本节讲解:`get_object_dict` / `get_object_fn` —— 查可用物体

在造任务之前,必须先确认"你要用的物体在代码库里存在"。

* **`get_object_dict()`** 返回一个字典:`{物体类别名: 对应的物体类}`。这里的**键**就是后续在场景/任务里引用的类别名,比如 `akita_black_bowl`、`wooden_cabinet`、`moka_pot`。打印它就能一览当前代码库注册了哪些可用物体。
* **`get_object_fn(category_name)`** 按类别名取出单个**物体类**(不是实例)。下一行展示"名字 -> 类"的映射关系。

为什么"物体要先注册"?因为这些类各自封装了网格模型、碰撞体、关节、可交互区域等物理属性,场景加载时就是靠类别名找到对应类来实例化的。可用物体之所以能被 `get_object_dict()` 搜到,是因为它们在定义时用了 `@register_object` 装饰器(见 `custom_object_example.ipynb`)。


In [3]:
category_name = "moka_pot"
object_cls = get_object_fn(category_name)
print(category_name, ": defined in the class ", object_cls)

moka_pot : defined in the class  <class 'libero.libero.envs.objects.turbosquid_objects.MokaPot'>


Similarly, you can retrieve the information about predicates.

### 本节讲解:谓词(predicate)—— 描述物体间的关系

和物体一样,**任务目标是用"谓词"表达的**。谓词就是"物体之间的关系 / 状态",例如上面代码里的 `on`(在…上)、以及后面会出现的 `In`(在…里面)、`Open`(打开)。

* **`get_predicate_fn_dict()`** 返回所有可用谓词的名字 -> 谓词函数(实现)的字典。
* **`get_predicate_fn(name)`** 按名字取出单个谓词函数。

这些谓词函数决定了:**如何判断一个目标状态是否被满足**(比如碗是否真的在柜子区域里)。因此下一步写 `goal_states` 时,里面用到的谓词名必须出现在 `predicate_dict` 里,否则任务无法被正确判定。


In [5]:
from libero.libero.envs.predicates import get_predicate_fn_dict, get_predicate_fn

predicate_dict = get_predicate_fn_dict()
print(predicate_dict)
print("=============")
predicate_name = "on"
print(get_predicate_fn(predicate_name))


{'true': <libero.libero.envs.predicates.base_predicates.TruePredicateFn object at 0x1377b7fd0>, 'false': <libero.libero.envs.predicates.base_predicates.FalsePredicateFn object at 0x1377b7ee0>, 'in': <libero.libero.envs.predicates.base_predicates.In object at 0x1377b7820>, 'on': <libero.libero.envs.predicates.base_predicates.On object at 0x1377b7580>, 'up': <libero.libero.envs.predicates.base_predicates.Up object at 0x1377b6b30>, 'printjointstate': <libero.libero.envs.predicates.base_predicates.PrintJointState object at 0x1377b6a40>, 'open': <libero.libero.envs.predicates.base_predicates.Open object at 0x1377b69b0>, 'close': <libero.libero.envs.predicates.base_predicates.Close object at 0x1377b5cc0>, 'turnon': <libero.libero.envs.predicates.base_predicates.TurnOn object at 0x1377b45e0>, 'turnoff': <libero.libero.envs.predicates.base_predicates.TurnOff object at 0x1377b4610>}


## 2. Define your own initial state distribution

### 本节讲解:`@register_mu` + `InitialSceneTemplates` —— 定义初始状态分布

这个 cell 是"任务生成"的核心:它定义了**一个场景模板**,并在其中规定每个物体的**初始位置采样区域**。

**`@register_mu(scene_type="kitchen")`** 是一个注册装饰器。它把类名 `KitchenScene1` 规范化成场景键 `kitchen_scene1`,注册到一个全局场景表里,后面 `register_task_info(scene_name="kitchen_scene1", ...)` 才能引用到它。`scene_type`(如 `"kitchen"`)用于给场景分类。

**`InitialSceneTemplates` 子类**必须提供两部分信息:

1. **`__init__` 里的数量清单**:
   * `fixture_num_info` —— 固定物数量,例如 `kitchen_table: 1`(工作台,即 workspace)、`wooden_cabinet: 1`(木柜)。
   * `object_num_info` —— 可动物体数量,例如 `akita_black_bowl: 1`、`plate: 1`。
   * `workspace_name` —— 指定哪个固定物是"工作台/桌面",物体的默认区域都挂在它上面。

2. **`define_regions(self)`**:给每个物体/固定物定义**初始化区域(region)**。`get_region_dict(...)` 的常见参数:
   * `region_centroid_xy` —— 区域中心在桌面局部坐标下的 (x, y);
   * `region_name` —— 区域名(后续 goal 里 `..._region` 会引用到,如 `wooden_cabinet_init_region`);
   * `target_name` —— 这个区域附着在哪个物体/固定物上(通常 `self.workspace_name`);
   * `region_half_len` —— 区域的半边长(越大,初始位姿采样范围越大 -> 初始状态分布越广);
   * `yaw_rotation` —— 朝向角的采样范围(如 `(np.pi, np.pi)` 表示固定朝向 π)。

**为什么要这样设计**:LIBERO 把"布局"和"任务"解耦——同一个 `InitialSceneTemplates` 定义了物体能出现的**初始状态分布**,同一场景可复用来生成大量不同目标的任务。这既是程序化造数据的基石,也是评测"策略对初始状态变化的泛化能力"的关键。


In [8]:
import numpy as np
from libero.libero.utils.bddl_generation_utils import get_xy_region_kwargs_list_from_regions_info
from libero.libero.utils.mu_utils import register_mu, InitialSceneTemplates
from libero.libero.utils.task_generation_utils import register_task_info, get_task_info, generate_bddl_from_task_info

@register_mu(scene_type="kitchen")
class KitchenScene1(InitialSceneTemplates):
    def __init__(self):

        fixture_num_info = {
            "kitchen_table": 1,
            "wooden_cabinet": 1,
        }

        object_num_info = {
            "akita_black_bowl": 1,
            "plate": 1,
        }

        super().__init__(
            workspace_name="kitchen_table",
            fixture_num_info=fixture_num_info,
            object_num_info=object_num_info
        )

    def define_regions(self):
        self.regions.update(
            self.get_region_dict(region_centroid_xy=[0.0, -0.30], 
                                 region_name="wooden_cabinet_init_region", 
                                 target_name=self.workspace_name, 
                                 region_half_len=0.01,
                                 yaw_rotation=(np.pi, np.pi))
        )

        self.regions.update(
            self.get_region_dict(region_centroid_xy=[0., 0.0], 
                                 region_name="akita_black_bowl_init_region", 
                                 target_name=self.workspace_name, 
                                 region_half_len=0.025)
        )

        self.regions.update(
            self.get_region_dict(region_centroid_xy=[0.0, 0.25], 
                                 region_name="plate_init_region", 
                                 target_name=self.workspace_name, 
                                 region_half_len=0.025)
        )
        self.xy_region_kwargs_list = get_xy_region_kwargs_list_from_regions_info(self.regions)

    @property
    def init_states(self):
        states = [
            ("On", "akita_black_bowl_1", "kitchen_table_akita_black_bowl_init_region"),
            ("On", "plate_1", "kitchen_table_plate_init_region"),
            ("On", "wooden_cabinet_1", "kitchen_table_wooden_cabinet_init_region")]
        return states

## 3. Define your own task goal

Now that you've defined the initial state distributions, you can specify a task goal based on the available objects and the potential goals it can acehive.

### 本节讲解:`register_task_info` —— 把"场景"变成"任务"

上一步用 `InitialSceneTemplates` 子类描述了**场景长什么样**(有哪些固定物、哪些物体、它们能出现在哪些区域)。这一步给场景**加上语言目标和成功条件**,从而定义一个具体任务。

`register_task_info` 的几个参数:

* **`language`**:任务的**自然语言指令**(示例里用了占位符 `"Your Language 1"`)。这是后续做语言条件策略(Language-conditioned policy)的输入文本。
* **`scene_name`**:引用注册过的 scene(`@register_mu(scene_type="kitchen")` 时,类名 `KitchenScene1` 会被转成 `kitchen_scene1`)。
* **`objects_of_interest`**:任务关注的物体列表(用来聚焦,可空)。
* **`goal_states`**:目标/成功条件,一个元组列表,每个元组是一个**谓词调用**。例如:
  * `("Open", "wooden_cabinet_1_top_region")` —— 谓词 `Open` 作用于柜子顶部区域;
  * `("In", "akita_black_bowl_1", "wooden_cabinet_1_top_region")` —— 碗 `In`(在…里面)柜子顶部区域。
  谓词名必须来自第 1 步 `get_predicate_fn_dict()` 里存在的那些。

注意示例里注册了**两个**任务:它们共用同一个 `scene_name`(场景布局相同),但 `goal_states` 不同(一个把碗放进顶部区域,另一个放进底部区域)。这就是"同场景、多目标"的批量造任务套路。

这些登记结果先暂存在 `libero.libero.utils.task_generation_utils.TASK_INFO`(命名元组 `TaskInfoTuple` 的列表)里,等下一步统一生成文件。


In [9]:
scene_name = "kitchen_scene1"
language = "Your Language 1"
register_task_info(language,
                    scene_name=scene_name,
                    objects_of_interest=["wooden_cabinet_1", "akita_black_bowl_1"],
                    goal_states=[("Open", "wooden_cabinet_1_top_region"), ("In", "akita_black_bowl_1", "wooden_cabinet_1_top_region")]
)

# Create another task with the same scene layout
scene_name = "kitchen_scene1"
language = "Your Language 2"
register_task_info(language,
                    scene_name=scene_name,
                    objects_of_interest=["wooden_cabinet_1", "akita_black_bowl_1"],
                    goal_states=[("Open", "wooden_cabinet_1_top_region"), ("In", "akita_black_bowl_1", "wooden_cabinet_1_bottom_region")]
)

The task goals will be temporarily saved in the variable `libero.libero.utils.task_generation_utils.TASK_INFO` in the format of namedtuple `libero.libero.utils.task_generation_utils.TaskInfoTuple`. This design aims to make it easy for batch creation of tasks.

### 本节讲解:`generate_bddl_from_task_info` 的产物

紧接上一步:`register_task_info(...)` 只是把任务"登记"到内存中的全局表 `TASK_INFO`(每个元素是一个 `TaskInfoTuple` 命名元组),**此时还没有任何文件生成**。这种"先收集、后统一生成"的设计,是为了方便一次批量造几百上千个任务。

下面这个 cell 才真正落盘:

* **`generate_bddl_from_task_info(folder=...)`**:遍历 `TASK_INFO` 里的每条任务,拿出它的 scene(初始场景模板)+ goal_states(目标状态),渲染成 BDDL 文本文件,写入 `folder`。
* **返回值**:`(bddl_file_names, failures)` —— 前者是所有成功生成的文件路径列表,后者是失败项(比如物体名写错、谓词参数不合法时会进入这里)。**一定要检查 `failures`**,它为空的案例才说明场景定义完全正确。
* **`objects_of_interest`**:列出这个任务里"值得关注"的物体。它主要用于数据采集/可视化时聚焦相关物体;示例里第二个任务把它留空 `[]` 也是合法的。

运行完后,`bddl_file_names[0]` 就是第一个任务的 BDDL 文件路径,下一步(最后一个 cell)会把它打印出来看内容。


In [10]:
# This is the default path to store all the pddl scene files. Here we store the files in the temporary folder. If you want to directly add files into the libero codebase, get the default path use the following commented lines:
# from libero.libero import get_libero_path
# YOUR_BDDL_FILE_PATH = get_libero_path("bddl_files")

YOUR_BDDL_FILE_PATH = "tmp/pddl_files"
bddl_file_names, failures = generate_bddl_from_task_info(folder=YOUR_BDDL_FILE_PATH)

print(bddl_file_names)

print("Encountered some failures: ", failures)


Succefully generated: 2
['tmp/pddl_files/KITCHEN_SCENE1_your_language_1.bddl', 'tmp/pddl_files/KITCHEN_SCENE1_your_language_2.bddl']
Encountered some failures:  []


Now you can see the content of the pddl file name. (Notice that we named our variable with bddl, since we are actually using the bddl package from Behavior. However, bddl is a subset of pddl, so we stick to the word PDDL for consistency in our paper writing and avoid confusion to the community.)

### 本节讲解:整条任务生成链路小结

到这里,procedural_creation_walkthrough 的四个步骤就串起来了。把整条链路连起来看:

```
get_object_dict() / get_object_fn()             # 1. 查可用物体(类别 -> 类)
get_predicate_fn_dict() / get_predicate_fn()    # 1. 查可用谓词(关系,如 on / In / Open)
        |
        v
@register_mu(scene_type=...)                    # 2. 用 InitialSceneTemplates 子类
class XxxScene(InitialSceneTemplates):          #    描述:固定物(fixture)+ 物体(object)
    def define_regions(...)                     #    以及它们各自的初始采样区域(region)
        |
        v
register_task_info(language,                    # 3. 用某个 scene + 目标状态(goal_states)
    scene_name=..., objects_of_interest=...,    #    注册一个"任务";结果暂存在 TASK_INFO
    goal_states=[("Open", ...), ("In", ...)])
        |
        v
generate_bddl_from_task_info(folder=...)        # 4. 批量把 TASK_INFO 渲染成 BDDL 文件
        |                                       #    返回 (成功文件名列表, 失败列表)
        v
open(bddl_file_names[0]).read()                 #    查看生成的 BDDL 文本
```

几个关键点:

* **物体 / 谓词必须先"存在"**:第 1 步先列出 `object_dict` 和 `predicate_dict`,是因为第 2、3 步里用到的名字(`akita_black_bowl`、`wooden_cabinet`、`on`、`In` ...)必须能在代码库里找到对应实现,否则场景加载或目标判定会失败。
* **`scene` 负责"在哪里",`task` 负责"做什么"**:同一个 scene 可以注册多个 task(示例里 `kitchen_scene1` 就注册了两个语言目标),这正是 LIBERO 能批量造大量同布局、不同目标任务的机制。
* **`region` 是初始状态的"采样区间"**:`get_region_dict(region_centroid_xy=..., region_half_len=..., yaw_rotation=...)` 定义的是一个矩形/朝向范围,生成器会在此基础上随机采样物体初始位姿,从而做**初始状态分布**的泛化。
* **BDDL vs PDDL**:代码里变量都叫 `bddl`,因为实际用的是 Behavior 的 `bddl` 包;bddl 是 pddl 的子集,论文里为统一表述仍沿用 "PDDL" 一词,别被命名搞混。
* **写到哪**:示例把文件写到 `tmp/pddl_files`(临时目录)。如果想直接进入代码库被 benchmark 复用,注释里给出了正确路径 `get_libero_path("bddl_files")`。


In [11]:
with open(bddl_file_names[0], "r") as f:
    content = f.read()
print(content)

(define (problem LIBERO_Kitchen_Tabletop_Manipulation)
  (:domain robosuite)
  (:language Your Language 1)
    (:regions
      (wooden_cabinet_init_region
          (:target kitchen_table)
          (:ranges (
              (-0.01 -0.31 0.01 -0.29)
            )
          )
          (:yaw_rotation (
              (3.141592653589793 3.141592653589793)
            )
          )
      )
      (akita_black_bowl_init_region
          (:target kitchen_table)
          (:ranges (
              (-0.025 -0.025 0.025 0.025)
            )
          )
          (:yaw_rotation (
              (0.0 0.0)
            )
          )
      )
      (plate_init_region
          (:target kitchen_table)
          (:ranges (
              (-0.025 0.225 0.025 0.275)
            )
          )
          (:yaw_rotation (
              (0.0 0.0)
            )
          )
      )
      (top_side
          (:target wooden_cabinet_1)
      )
      (top_region
          (:target wooden_cabinet_1)
      )
      (middl